# Scanpy Extra: (For quick testing)

In [37]:
# This cell is labelled 'paramters' to work with papermill remotely
# Papermill will overwite the default local plate value below with whatever is passed to 
# the -p flag in the snakerule shell script
#os.system("conda activate eqtl_study") use this locally if using VScode
plate = 'plate1'
plate = globals().get("plate")
print(f"Processing plate: {plate}")

Processing plate: plate1


### Set Variables

In [ ]:
# Import custom utility packages, lists and functions
import sys
import os
if os.path.exists('/scratch/'):
    root_dir = '/scratch/c.c1477909/eQTL_study_2025/'
else:
    root_dir = '/Users/darren/Desktop/eQTL_study_2025/'
        
sys.path.append(root_dir + 'workflow/scripts/')

from init_env import *
from anndata_utils import *
from gene_lists import *

# Set variables
resolutions = [0.1, 0.2, 0.3] 

### Load L1 data

In [ ]:
# Initialize the environment and get all paths and logger
logger, root_dir, sheets_dir, plate_path, scanpy_dir, report_dir = initialize_env(plate)
logger.info("Loading data ...")
adata = sc.read(scanpy_dir + f'adata_clusters.h5ad')

### Set cluster IDs

In [ ]:
logger.info("Setting cluster IDs ...")
# Set cluster names
cluster_anns = {
    
    '0': 'ExN-UL',
    '1': 'ExN-DL',
    '2': 'RG',
    '3': 'InN',
    '4': 'Endo-Peri',
    '5': 'OPC',
    '6': 'MG'
}

custom_palette = {
    'RG': '#FF5959',
    'ExN-UL': '#00B6EB',
    'InN': '#3CBB75FF',
    'ExN-DL': '#CEE5FD',
    'Endo-Peri': '#B200ED',
    'MG': '#F58231',
    'OPC': '#FDE725FF'
}

final_genes = [
    "CUX2", "SATB2",      
    "TLE4",                 
    "GAD1", "GAD2",                 
    "GLI3", "PRDM16", "PAX6",  
    "COL4A1", "FN1",                              
    "PDGFRA",                                      
    "C3",                                          
]


# Create a new column in adata.obs with cell type names
adata.obs['cell_type'] = adata.obs['leiden_harmony_0.1'].map(cluster_anns)

### Create UMAP - manuscript

In [ ]:

import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap

# -----------------------------
# Custom light → dark blue colormap
# -----------------------------
blue_cmap = LinearSegmentedColormap.from_list(
    "light_dark_blue",
    ["#f7fbff", "#08306b"]
)

# -----------------------------
# Configuration
# -----------------------------
n_genes = len(final_genes)

# Feature plot grid geometry
ncols = 3
nrows = math.ceil(n_genes / ncols)

# Nature Genetics: double-column figure
fig_width = 9  # inches (174 mm)
fig_height = 4.5 + (nrows * 0.95)

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 9,
    "axes.titlesize": 12,
})

# -----------------------------
# Create figure and layout
# -----------------------------
fig = plt.figure(figsize=(fig_width, fig_height))
gs = gridspec.GridSpec(
    nrows=nrows,
    ncols=ncols + 2,
    figure=fig,
    width_ratios=[1.4, 1.4] + [1] * ncols,
    wspace=0.35,
    hspace=0.35
)

# -----------------------------
# Panel A: Cell type UMAP
# -----------------------------
ax_main = fig.add_subplot(gs[:, :2])

sc.pl.umap(
    adata,
    color="cell_type",
    ax=ax_main,
    legend_loc="on data",
    legend_fontsize=11,
    legend_fontoutline=3,
    frameon=False,
    title="",
    show=False
)

ax_main.set_title("A", loc="left", fontweight="bold", fontsize=14)

# -----------------------------
# Panel B: Feature plots
# -----------------------------
feature_axes = []
for i, gene in enumerate(final_genes):
    row = i // ncols
    col = i % ncols
    ax = fig.add_subplot(gs[row, col + 2])
    feature_axes.append(ax)

    sc.pl.umap(
        adata,
        color=gene,
        ax=ax,
        vmin=0,
        vmax="p99",
        cmap=blue_cmap,
        frameon=False,
        title=gene,
        show=False
    )

    ax.title.set_fontsize(12)

# Panel label B
feature_axes[0].set_title("B", loc="left", fontweight="bold", fontsize=14)


# -----------------------------
# Save outputs
# -----------------------------
plot_dir = '../results/13MANUSCRIPT_PLOTS_TABLES/'
plt.savefig(plot_dir + "umaps_manuscript.png", dpi=300, bbox_inches="tight")
plt.savefig(plot_dir + "umaps_manuscript.pdf", bbox_inches="tight")
plt.close(fig)
